# Speech Denoising — Notebook 06: Evaluation (PESQ + STOI)

Objective evaluation of the trained model using two industry-standard metrics:

**PESQ** (Perceptual Evaluation of Speech Quality, ITU-T P.862) — measures signal quality. Range: -0.5 (worst) → 4.5 (best).

**STOI** (Short-Time Objective Intelligibility) — measures speech intelligibility. Range: 0 (worst) → 1 (best).

**Methodology:** Run inference on all test chunks, compute both metrics against clean reference, compare against noisy baseline (no model).

## 1. Imports

In [ ]:
import numpy as np
import torch
from torch import nn
import librosa as lb
from IPython.display import Audio
from pesq import pesq
from pystoi import stoi

## 2. Load Test Data

Test chunks preprocessed in notebook 01 — 1-second segments (16,000 samples) not seen during training.

In [ ]:
test_noisy_chunks = np.load('data/test_noisy_chunks.npy')
test_clean_chunks = np.load('data/test_clean_chunks.npy')

## 3. Model Architecture & Load Weights

In [ ]:
class DenoisingModelDilated(nn.Module):
    def __init__(self):
        super().__init__()
        self.relu = nn.LeakyReLU()
        self.layer1 = nn.Conv1d(1, 16, kernel_size=3, dilation=1, padding=1)
        self.layer2 = nn.Conv1d(16, 32, kernel_size=3, dilation=2, padding=2)
        self.layer3 = nn.Conv1d(32, 16, kernel_size=3, dilation=4, padding=4)
        self.layer4 = nn.Conv1d(16, 1, kernel_size=3, dilation=8, padding=8)
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        x = self.relu(x)
        x = self.layer3(x)
        x = self.relu(x)
        x = self.layer4(x)
        x = x.squeeze(1)
        return x

In [ ]:
model = DenoisingModelDilated()
model.load_state_dict(torch.load('denoising_cnn_spectral_loss_4layers_4epochs.pth'))
model.eval()

## 4. Sanity Check — Single Chunk

Verify inference pipeline on one chunk before running the full evaluation loop.

In [ ]:
chunk_tensor = torch.from_numpy(test_noisy_chunks[0]).unsqueeze(0)
pred_clean_chunk = model(chunk_tensor).detach().numpy().squeeze(0)

pesq_score_single = pesq(16000, test_clean_chunks[0], pred_clean_chunk, 'wb')
stoi_score_single = stoi(test_clean_chunks[0], pred_clean_chunk, 16000)
print(f"PESQ: {pesq_score_single:.4f}")
print(f"STOI: {stoi_score_single:.4f}")

## 5. Full Evaluation — Model PESQ

`valid_indices` tracks chunks where PESQ detects speech. Used for fair comparison with baseline.

`NoUtterancesError` is caught and skipped (silence/noise-only chunks).

In [ ]:
pesq_score_list = []
valid_indices = []
for i, chunk in enumerate(test_noisy_chunks):
    chunk_tensor = torch.from_numpy(chunk).unsqueeze(0)
    pred_clean_chunk = model(chunk_tensor).detach().numpy().squeeze(0)
    try:
        pesq_score = pesq(16000, test_clean_chunks[i], pred_clean_chunk, 'wb')
        pesq_score_list.append(pesq_score)
        valid_indices.append(i)
    except Exception:
        pass

print(f"Mean PESQ (model): {np.mean(pesq_score_list):.4f}")
print(f"Chunks scored: {len(pesq_score_list)}")

## 6. Full Evaluation — Noisy Baseline PESQ

Same `valid_indices` — identical chunks for fair comparison.

In [ ]:
pesq_noisy_score_list = []
for i in valid_indices:
    chunk = test_noisy_chunks[i]
    try:
        pesq_score = pesq(16000, test_clean_chunks[i], chunk, 'wb')
        pesq_noisy_score_list.append(pesq_score)
    except Exception:
        pass

print(f"Mean PESQ (noisy baseline): {np.mean(pesq_noisy_score_list):.4f}")
print(f"Chunks scored: {len(pesq_noisy_score_list)}")

## 7. Full Evaluation — Model STOI

STOI does not throw on silence — no try/except needed.

In [ ]:
stoi_score_list = []
for i, chunk in enumerate(test_noisy_chunks):
    chunk_tensor = torch.from_numpy(chunk).unsqueeze(0)
    pred_clean_chunk = model(chunk_tensor).detach().numpy().squeeze(0)
    stoi_score = stoi(test_clean_chunks[i], pred_clean_chunk, 16000)
    stoi_score_list.append(stoi_score)

print(f"Mean STOI (model): {np.mean(stoi_score_list):.4f}")

## 8. Full Evaluation — Noisy Baseline STOI

In [ ]:
stoi_noisy_score_list = []
for i, chunk in enumerate(test_noisy_chunks):
    stoi_score = stoi(test_clean_chunks[i], chunk, 16000)
    stoi_noisy_score_list.append(stoi_score)

print(f"Mean STOI (noisy baseline): {np.mean(stoi_noisy_score_list):.4f}")

## 9. Results Summary

| | PESQ | STOI |
|---|---|---|
| Noisy input (no model) | 2.10 | 0.90 |
| Model output | 2.04 | 0.87 |

**Observation:** Model slightly underperforms the noisy baseline on both metrics. This is expected for a shallow dilated CNN trained for only 4 epochs. Both PESQ and STOI measure mathematical similarity to the clean reference — informal listening tests suggest the model does reduce perceived noise despite lower scores.

**Why the gap:** The model was trained on MSE in spectral domain (spectral loss), not directly optimized for PESQ or STOI. Architectures like U-Net or DeepFilterNet, trained with perceptual losses, typically reach PESQ > 2.5 on this dataset.

**Next steps:** U-Net architecture, more epochs, perceptual loss functions.